In [1]:
import requests

In [2]:
api_url = "https://115.159.159.82/api/devices"

r = requests.get(
    api_url,
    params={"page": 1, "page_size": 50},
    timeout=10
)

In [3]:
print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [4]:
data = r.json()

print(type(data))
print(list(data.keys()))

<class 'dict'>
['page', 'page_size', 'total', 'items']


In [5]:
print("当前页   :", data["page"])
print("每页条数 :", data["page_size"])
print("设备总数 :", data["total"])
print("本页条数 :", len(data["items"]))

当前页   : 1
每页条数 : 50
设备总数 : 5000
本页条数 : 50


In [6]:
import json

print(json.dumps(data["items"][0], ensure_ascii=False, indent=2))

{
  "device_id": "D00000001",
  "company_id": "C000001",
  "device_name": "循环泵-000001-001",
  "device_type": "pump",
  "model": "SP-PUM-800",
  "zone": "智造一区",
  "install_date": "2021-09-19",
  "rated_power_kw": 71.2,
  "sampling_interval_s": 30,
  "protocol": "mqtt",
  "status": "running",
  "updated_at": "2026-09-01T03:58:00"
}


In [8]:
import time

all_devices = []
page = 1

while True:
    r = requests.get(api_url, params={"page": page, "page_size": 50}, timeout=10)
    d = r.json()
    items = d["items"]
    all_devices.extend(items)
    print(f"第 {page:3d} 页：本页 {len(items)} 条，累计 {len(all_devices)} / {d['total']}")

    if not items or len(all_devices) >= d["total"]:
        break
    page += 1
    time.sleep(0.2)

print("最终条数:", len(all_devices))

第   1 页：本页 50 条，累计 50 / 5000
第   2 页：本页 50 条，累计 100 / 5000
第   3 页：本页 50 条，累计 150 / 5000
第   4 页：本页 50 条，累计 200 / 5000
第   5 页：本页 50 条，累计 250 / 5000
第   6 页：本页 50 条，累计 300 / 5000
第   7 页：本页 50 条，累计 350 / 5000
第   8 页：本页 50 条，累计 400 / 5000
第   9 页：本页 50 条，累计 450 / 5000
第  10 页：本页 50 条，累计 500 / 5000
第  11 页：本页 50 条，累计 550 / 5000
第  12 页：本页 50 条，累计 600 / 5000
第  13 页：本页 50 条，累计 650 / 5000
第  14 页：本页 50 条，累计 700 / 5000
第  15 页：本页 50 条，累计 750 / 5000
第  16 页：本页 50 条，累计 800 / 5000
第  17 页：本页 50 条，累计 850 / 5000
第  18 页：本页 50 条，累计 900 / 5000
第  19 页：本页 50 条，累计 950 / 5000
第  20 页：本页 50 条，累计 1000 / 5000
第  21 页：本页 50 条，累计 1050 / 5000
第  22 页：本页 50 条，累计 1100 / 5000
第  23 页：本页 50 条，累计 1150 / 5000
第  24 页：本页 50 条，累计 1200 / 5000
第  25 页：本页 50 条，累计 1250 / 5000
第  26 页：本页 50 条，累计 1300 / 5000
第  27 页：本页 50 条，累计 1350 / 5000
第  28 页：本页 50 条，累计 1400 / 5000
第  29 页：本页 50 条，累计 1450 / 5000
第  30 页：本页 50 条，累计 1500 / 5000
第  31 页：本页 50 条，累计 1550 / 5000
第  32 页：本页 50 条，累计 1600 / 5000
第  33 页：本页 50 条，累计 1650 / 50

In [9]:
from collections import Counter

ids = [d["device_id"] for d in all_devices]
dup = [k for k, v in Counter(ids).items() if v > 1]

print("设备总数 :", len(ids))
print("唯一 ID 数 :", len(set(ids)))
print("重复 ID  :", len(dup))


设备总数 : 5000
唯一 ID 数 : 5000
重复 ID  : 0


In [11]:
import os

print("当前工作目录:", os.getcwd())

with open("data/devices.json", "w", encoding="utf-8") as f:
    json.dump(all_devices, f, ensure_ascii=False, indent=2)

print("已保存 data/devices.json，大小:", os.path.getsize("data/devices.json"), "字节")


当前工作目录: /home/pwuser/workspace
已保存 data/devices.json，大小: 1907611 字节


In [12]:
from bs4 import BeautifulSoup

comp_url = "https://115.159.159.82/companies"

r = requests.get(comp_url, timeout=10)
print("status:", r.status_code)

soup = BeautifulSoup(r.text, "html.parser")
cards = soup.select("article.company-card")
print("第 1 页企业数:", len(cards))


status: 200
第 1 页企业数: 20


In [13]:
print(cards[0].prettify())


<article class="company-card" data-company-id="C000001">
 <div class="company-card-top">
  <span class="mono">
   C000001
  </span>
  <span class="badge active">
   运营中
  </span>
 </div>
 <h2>
  <a href="/companies/C000001">
   临港精工001有限公司
  </a>
 </h2>
 <dl>
  <div>
   <dt>
    所属行业
   </dt>
   <dd>
    汽车零部件
   </dd>
  </div>
  <div>
   <dt>
    所在区域
   </dt>
   <dd>
    智造一区
   </dd>
  </div>
  <div>
   <dt>
    员工规模
   </dt>
   <dd>
    88 人
   </dd>
  </div>
  <div>
   <dt>
    设备数量
   </dt>
   <dd>
    6 台
   </dd>
  </div>
 </dl>
 <a class="detail-link" href="/companies/C000001">
  查看企业档案
  <span aria-hidden="true">
   →
  </span>
 </a>
</article>



In [14]:
def parse_card(card):
    def dd(field):
        for item in card.select("dl > div"):
            t = item.select_one("dt")
            d = item.select_one("dd")
            if t and d and t.get_text(strip=True) == field:
                return d.get_text(strip=True)

    return {
        "company_id": card["data-company-id"],
        "name": card.select_one("h2 a").get_text(strip=True),
        "status": card.select_one(".badge").get_text(strip=True),
        "industry": dd("所属行业"),
        "zone": dd("所在区域"),
        "employee_scale": dd("员工规模"),
        "device_count": int(dd("设备数量").replace("台", "").strip()),
    }

companies = [parse_card(c) for c in cards]
print("解析出企业数:", len(companies))
print(json.dumps(companies[0], ensure_ascii=False, indent=2))


解析出企业数: 20
{
  "company_id": "C000001",
  "name": "临港精工001有限公司",
  "status": "运营中",
  "industry": "汽车零部件",
  "zone": "智造一区",
  "employee_scale": "88 人",
  "device_count": 6
}


In [17]:
from urllib.parse import urljoin

companies_all = []
current_url = comp_url
page_no = 1

while current_url and page_no <= 3:
    r = requests.get(current_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    cards = soup.select("article.company-card")
    companies_all.extend(parse_card(c) for c in cards)
    print(f"第 {page_no} 页：{len(cards)} 家，累计 {len(companies_all)}")

    next_link = None
    for a in soup.select("nav.pagination a"):
        if a.get_text(strip=True) == "下一页":
            next_link = a
            break

    current_url = urljoin(current_url, next_link["href"]) if next_link else None
    page_no += 1
    time.sleep(0.2)

companies = companies_all
print("三页合计:", len(companies))


第 1 页：20 家，累计 20
第 2 页：20 家，累计 40
第 3 页：20 家，累计 60
三页合计: 60


In [18]:
from collections import Counter

ids = [c["company_id"] for c in companies]
dup = [k for k, v in Counter(ids).items() if v > 1]

print("企业总数 :", len(ids))
print("唯一 ID 数 :", len(set(ids)))
print("重复 ID  :", len(dup))


企业总数 : 60
唯一 ID 数 : 60
重复 ID  : 0


In [19]:
with open("data/companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("已保存 data/companies.json，大小:", os.path.getsize("data/companies.json"), "字节")


已保存 data/companies.json，大小: 13490 字节


In [20]:
from collections import Counter

company_ids = [c["company_id"] for c in companies]
device_ids = [d["device_id"] for d in all_devices]

device_counts = Counter(x["company_id"] for x in all_devices)

mismatch = 0
for c in companies:
    expected = c["device_count"]
    actual = device_counts.get(c["company_id"], 0)
    if expected != actual:
        mismatch += 1
        print("不一致:", c["company_id"], c["name"],
              "页面设备数:", expected, "实际采集数:", actual)

print("=" * 40)
print("Company 总数                 :", len(company_ids))
print("唯一 company_id 数           :", len(set(company_ids)))
print("Device 总数                  :", len(device_ids))
print("唯一 device_id 数            :", len(set(device_ids)))
print("Company–Device 不一致企业数  :", mismatch)


Company 总数                 : 60
唯一 company_id 数           : 60
Device 总数                  : 5000
唯一 device_id 数            : 5000
Company–Device 不一致企业数  : 0
